# Решения: CLI и контракт эксперимента

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

SCRIPT = Path("train_cli.py")
DATA_PATH = Path("bank_marketing_slim.csv")
assert SCRIPT.exists() and DATA_PATH.exists()


## Урок. 1. Команда

In [ ]:
cmd = [sys.executable, str(SCRIPT), '--data', str(DATA_PATH), '--threshold', '0.45']
assert len(cmd) == 6


## Урок. 2–3. Процесс и JSON

In [ ]:
proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
assert proc.returncode == 0, proc.stderr
metrics = json.loads(proc.stdout)
expected_keys = {'threshold', 'duration_in_features', 'accuracy', 'precision', 'recall', 'f1'}
assert set(metrics) == expected_keys and metrics['duration_in_features'] is False


## Урок. 4. Runner

In [ ]:
def run_experiment(threshold, seed=63):
    command = [
        sys.executable, str(SCRIPT), "--data", str(DATA_PATH),
        "--threshold", str(threshold), "--seed", str(seed),
    ]
    completed = subprocess.run(command, capture_output=True, text=True, check=False)
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr or completed.stdout)
    return json.loads(completed.stdout)
trial = run_experiment(0.5)
assert trial['threshold'] == 0.5


## Урок. 5–6. Серия запусков

In [ ]:
import pandas as pd
thresholds = [0.25, 0.35, 0.45, 0.55, 0.65]
runs = [run_experiment(t) for t in thresholds]
run_table = pd.DataFrame(runs)
assert run_table['recall'].is_monotonic_decreasing


## Урок. 7. Acceptance gate

In [ ]:
repeat = run_experiment(0.45)
checks = {'process_ok': proc.returncode == 0, 'json_contract': set(metrics) == expected_keys, 'duration_forbidden': metrics['duration_in_features'] is False, 'metrics_in_range': all(0 <= metrics[n] <= 1 for n in ('accuracy', 'precision', 'recall', 'f1')), 'deterministic': repeat == metrics}
assert set(checks.values()) == {True}


## Урок. 8. Диагностика ошибки

In [ ]:
bad_cmd = [sys.executable, str(SCRIPT), '--data', 'missing.csv', '--threshold', '0.5']
bad_proc = subprocess.run(bad_cmd, capture_output=True, text=True, check=False)
error_text = bad_proc.stderr + bad_proc.stdout
assert bad_proc.returncode != 0 and len(error_text) > 20


## Урок. 9. Отчёт

In [ ]:
best = run_table.loc[run_table["f1"].idxmax()]
CLI_REPORT = (f"Команда CLI принимает --data, --threshold и --seed; все пять запусков вернули JSON. "
f"Лучший F1={best.f1:.3f} при threshold={best.threshold:.2f}. Поле duration_in_features во всех результатах false, "
"поэтому guard против duration сработал. Ошибочный путь завершился ненулевым кодом и дал stderr, пригодный для диагностики. Повтор с тем же seed детерминирован.")
assert len(CLI_REPORT) >= 240


## ДЗ. A1. Универсальный runner

In [ ]:
result = run_experiment(0.4, 64)
assert result['threshold'] == 0.4


## ДЗ. A2. Сетка запусков

In [ ]:
rows = []
for threshold in (0.3, 0.5, 0.7):
    for seed in (11, 22, 33):
        row = run_experiment(threshold, seed)
        row['seed'] = seed
        rows.append(row)
grid = pd.DataFrame(rows)
assert len(grid) == 9 and not grid['duration_in_features'].any()


## ДЗ. A3. Устойчивость

In [ ]:
stability = grid.groupby('threshold').agg(f1_min=('f1', 'min'), f1_max=('f1', 'max'), f1_mean=('f1', 'mean'))
assert len(stability) == 3


## ДЗ. Challenge. Gate

In [ ]:
def acceptance(result):
    required = {'threshold', 'duration_in_features', 'accuracy', 'precision', 'recall', 'f1'}
    return required <= set(result) and result['duration_in_features'] is False and all(0 <= result[n] <= 1 for n in ('accuracy', 'precision', 'recall', 'f1'))

assert all(acceptance(row) for row in rows) and not acceptance({'duration_in_features': True})


## ДЗ. Challenge. Инженерная записка

In [ ]:
ENGINEERING_NOTE = ("CLI отделяет параметры запуска от кода и позволяет повторить эксперимент одной командой. seed фиксирует split, "
"но сетка seed показывает устойчивость метрик. JSON-контракт проверяется автоматически; duration обязан оставаться false. "
"Ненулевой return code нельзя игнорировать: stderr сохраняет причину ошибки пути или аргумента. Такой runner можно включить в следующий gate, не копируя notebook-состояние и не читая вывод вручную.")
assert len(ENGINEERING_NOTE) >= 280
